# 🎙️ VoiceTyper: Audio Denoising, Volume Gain & Speech Enhancement Prototype

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/burubur/voicetyper/blob/main/notebooks/noise_reduction_prototype.ipynb)

This interactive prototype allows you to inspect, listen, and verify speech enhancement algorithms on your own VoiceTyper recordings:
1. **Classical DSP**: 80Hz High-Pass Filter + Stationary Spectral Subtraction (cuts 50/60Hz hum, desk bumps, mic rumble, background hiss).
2. **Adaptive Speech Gain & AGC Normalization**: Solves low volume issues by boosting speech to -1.0 dBFS headroom with soft-knee limiting (+6dB to +18dB clean boost).
3. **Neural Voice Activity Detection (Silero VAD)**: Isolates pure speech segments and strips dead silence pauses.
4. **Interactive Audio Players & Whisper Comparison**: Listen to each stage and compare transcription accuracy.

In [ ]:
# 1. Install dependencies
!pip install -q noisereduce scipy soundfile matplotlib librosa openai-whisper torch torchaudio

In [ ]:
# 2. Load Audio File (Upload your VoiceTyper .wav file or generate synthetic demo audio)
import io
import os
import soundfile as sf
import librosa
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

sr = 16000
try:
    from google.colab import files
    print("📤 Upload a VoiceTyper .wav file (from ~/.voicetyper/conversation/ or your Mac):")
    uploaded = files.upload()
    if uploaded:
        audio_path = list(uploaded.keys())[0]
        raw_audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    else:
        raise ValueError("No file uploaded, generating quiet synthetic speech demo.")
except Exception as e:
    print(f"Note: {e}\nGenerating quiet synthetic speech with ambient noise (simulating low mic volume)...")
    t = np.linspace(0, 3, 3 * sr)
    # Quiet speech signal (peak ~0.15 = -16.5 dBFS) + 60Hz hum + hiss
    speech = 0.15 * np.sin(2 * np.pi * 220 * t) * (np.sin(2 * np.pi * 2 * t) > 0)
    hum = 0.04 * np.sin(2 * np.pi * 60 * t)
    noise = 0.02 * np.random.normal(0, 1, len(t))
    raw_audio = (speech + hum + noise).astype(np.float32)
    audio_path = "demo_audio.wav"
    sf.write(audio_path, raw_audio, sr)

raw_peak = np.max(np.abs(raw_audio))
raw_rms = np.sqrt(np.mean(raw_audio**2))
print(f"\n✓ Loaded Audio: {audio_path} ({len(raw_audio)/sr:.2f}s @ {sr}Hz)")
print(f"  • Raw Peak Level: {raw_peak:.4f} ({20*np.log10(max(1e-5, raw_peak)):.1f} dBFS)")
print(f"  • Raw RMS Energy: {raw_rms:.4f} ({20*np.log10(max(1e-5, raw_rms)):.1f} dBFS)")

print("\n🎧 1. Original Raw Input:")
ipd.display(ipd.Audio(raw_audio, rate=sr))

In [ ]:
# 3. Stage 1: Classical DSP Denoising (High-Pass + Spectral Subtraction)
from scipy import signal
import noisereduce as nr

def apply_dsp_denoising(audio, sr=16000, strength=0.75):
    # 80Hz High-Pass Butterworth filter (strips AC hum & desk rumble)
    sos = signal.butter(4, 80, btype='highpass', fs=sr, output='sos')
    filtered = signal.sosfilt(sos, audio)
    
    # Stationary Spectral Subtraction
    denoised = nr.reduce_noise(
        y=filtered, 
        sr=sr, 
        prop_decrease=strength, 
        stationary=True,
        n_fft=512,
        win_length=512,
        hop_length=256
    )
    return denoised

dsp_audio = apply_dsp_denoising(raw_audio, sr)
sf.write("dsp_cleaned.wav", dsp_audio, sr)

dsp_peak = np.max(np.abs(dsp_audio))
dsp_rms = np.sqrt(np.mean(dsp_audio**2))
print(f"✓ Denoised Peak: {dsp_peak:.4f} ({20*np.log10(max(1e-5, dsp_peak)):.1f} dBFS) | RMS: {dsp_rms:.4f} ({20*np.log10(max(1e-5, dsp_rms)):.1f} dBFS)")
print("🎧 2. Denoised Audio (Without Gain Normalization - Notice Low Volume):")
ipd.display(ipd.Audio(dsp_audio, rate=sr))

In [ ]:
# 4. Stage 2: Adaptive Speech Gain & Dynamic Normalization (AGC Solution)
def apply_adaptive_gain(audio, target_peak=0.89, max_gain=5.0):
    """
    Boosts speech audio to optimal -1.0 dBFS (0.89) peak headroom
    with soft-knee tanh compression to eliminate clipping.
    """
    peak = np.max(np.abs(audio))
    if peak < 1e-4:
        return audio, 1.0
    
    # Calculate necessary linear gain multiplier
    gain = min(target_peak / peak, max_gain)
    boosted = audio * gain
    
    # Soft tanh limiting to smoothly handle transient peaks without distortion
    normalized = np.tanh(boosted) * 0.95
    return normalized.astype(np.float32), gain

boosted_audio, gain_factor = apply_adaptive_gain(dsp_audio, target_peak=0.89, max_gain=5.0)
sf.write("boosted_cleaned.wav", boosted_audio, sr)

boosted_peak = np.max(np.abs(boosted_audio))
boosted_rms = np.sqrt(np.mean(boosted_audio**2))

print(f"✓ Applied Gain Boost: {gain_factor:.2f}x (+{20*np.log10(gain_factor):.1f} dB)")
print(f"  • Boosted Peak Level: {boosted_peak:.4f} ({20*np.log10(boosted_peak):.1f} dBFS)")
print(f"  • Boosted RMS Level : {boosted_rms:.4f} ({20*np.log10(boosted_rms):.1f} dBFS)")

print("\n🎧 3. Denoised + Adaptive Gain Boost (Loud, Crisp & Clear):")
ipd.display(ipd.Audio(boosted_audio, rate=sr))

In [ ]:
# 5. Stage 3: Neural Voice Activity Detection (Silero VAD)
import torch

model, utils = torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False
)
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

wav_tensor = torch.from_numpy(boosted_audio).float()
speech_timestamps = get_speech_timestamps(wav_tensor, model, sampling_rate=16000, threshold=0.45)

if speech_timestamps:
    vad_audio_tensor = collect_chunks(speech_timestamps, wav_tensor)
    vad_audio = vad_audio_tensor.numpy()
else:
    print("⚠️ No speech detected by VAD!")
    vad_audio = boosted_audio

sf.write("vad_cleaned.wav", vad_audio, sr)
print(f"✓ Extracted {len(speech_timestamps)} speech chunks ({len(vad_audio)/sr:.2f}s vs {len(raw_audio)/sr:.2f}s total duration)")
print("🎧 4. Silero VAD (Speech Only, Zero Pauses):")
ipd.display(ipd.Audio(vad_audio, rate=sr))

In [ ]:
# 6. Visual Waveform & Spectrogram Comparison
fig, axes = plt.subplots(4, 2, figsize=(14, 10), sharex=True)

def plot_audio_spec(audio, title, row, color='#6366f1'):
    time_axis = np.linspace(0, len(audio)/sr, len(audio))
    axes[row, 0].plot(time_axis, audio, color=color, alpha=0.85)
    axes[row, 0].set_title(f"{title} - Waveform (Peak: {np.max(np.abs(audio)):.2f})")
    axes[row, 0].set_ylabel("Amplitude")
    axes[row, 0].set_ylim(-1.05, 1.05)
    axes[row, 0].grid(True, alpha=0.3)
    
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=axes[row, 1], cmap='magma')
    axes[row, 1].set_title(f"{title} - Spectrogram")
    axes[row, 1].set_ylim(0, 8000)

plot_audio_spec(raw_audio, "1. Original Raw Audio (Low Gain)", 0, color='#94a3b8')
plot_audio_spec(dsp_audio, "2. Denoised (Unnormalized)", 1, color='#f59e0b')
plot_audio_spec(boosted_audio, f"3. Denoised + Adaptive Gain ({gain_factor:.1f}x Boost)", 2, color='#10b981')
plot_audio_spec(vad_audio, "4. Silero VAD Speech Only", 3, color='#8b5cf6')

plt.tight_layout()
plt.show()

In [ ]:
# 7. Transcription Benchmark with Whisper
import whisper

print("Loading Whisper base.en model...")
whisper_model = whisper.load_model("base.en")

print("\n================ Transcription Comparison ================")
res_raw = whisper_model.transcribe(raw_audio)
print(f"1. Raw Audio            : \"{res_raw['text'].strip()}\"")

res_dsp = whisper_model.transcribe(dsp_audio)
print(f"2. Denoised Only        : \"{res_dsp['text'].strip()}\"")

res_boosted = whisper_model.transcribe(boosted_audio)
print(f"3. Denoised + Gain Boost: \"{res_boosted['text'].strip()}\"")

res_vad = whisper_model.transcribe(vad_audio)
print(f"4. VAD Cleaned Speech   : \"{res_vad['text'].strip()}\"")
print("===========================================================")